In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


df = pd.read_csv('Crashes.csv')

df_clean = df.dropna(subset=['CRSH_LEVL'])
weather_features = ['temperature_2m (°C)', 'precipitation (mm)', 'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)']
features = ['X', 'Y', 'DATE_VAL_YEAR', 'DATE_VAL_MONTH', 'DATE_VAL_DAY', 'DAY_OF_WEEK', 'MILT_TIME', 'CRSH_TYPE_CD'] + weather_features

X_data = df_unified[features]

Y_data = df_unified['CRSH_LEVL']



print("AI Features Preview:")
print(X_data.head())
print(Y_data.head())

AI Features Preview:
              X              Y  DATE_VAL_YEAR  DATE_VAL_MONTH  DATE_VAL_DAY  \
0  1.459633e+06  574471.294491           2013               7            26   
1  1.485534e+06  569680.825398           2013               7            27   
2  1.453856e+06  540117.288725           2013               7            27   
3  1.433454e+06  551124.237839           2013               7            27   
4  1.432444e+06  510347.904366           2013               7            27   

   DAY_OF_WEEK  MILT_TIME  CRSH_TYPE_CD  temperature_2m (°C)  \
0            6       2206             6                 23.5   
1            7        136             3                 21.6   
2            7        215            14                 21.2   
3            7        245            29                 21.2   
4            7        436             1                 20.4   

   precipitation (mm)  wind_speed_10m (km/h)  wind_gusts_10m (km/h)  
0                 0.0                    3.3     

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

x_train, x_test, y_train, y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)


print("Training the AI model now...")
model = RandomForestClassifier(n_estimators=100, class_weight = 'balanced', max_depth = 12, random_state=42, n_jobs=-1)


model.fit(x_train, y_train)
print("Training complete!")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Test Accuracy: {accuracy * 100:.2f}%")


print("\nDetailed Performance Report:")
print(classification_report(y_test, y_pred))

Training the AI model now...
Training complete!

Model Test Accuracy: 54.46%

Detailed Performance Report:
              precision    recall  f1-score   support

         1.0       0.04      0.19      0.06       183
         2.0       0.04      0.06      0.05       310
         3.0       0.16      0.31      0.21      5360
         4.0       0.28      0.56      0.38     19936
         5.0       0.85      0.56      0.68     73779
         6.0       0.00      0.00      0.00         1

    accuracy                           0.54     99569
   macro avg       0.23      0.28      0.23     99569
weighted avg       0.70      0.54      0.59     99569



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
import numpy as np


# 1. Type in your custom coordinates, dates, and times here to test the AI
custom_crash_scenario = {
    'X': 1460000,
    'Y': 550100,
    'DATE_VAL_YEAR': 2016,
    'DATE_VAL_MONTH': 8,
    'DATE_VAL_DAY': 10,
    'DAY_OF_WEEK': 4,
    'MILT_TIME': 250,
    'CRSH_TYPE_CD': 30,
    'temperature_2m (°C)': 2,     # Below freezing (27°F)
    'precipitation (mm)': 5,       # Active freezing rain/snow
    'wind_speed_10m (km/h)': 50.0,
    'wind_gusts_10m (km/h)': 49.0,
}

# 2. Convert your inputs into the exact format the AI expects
input_df = pd.DataFrame([custom_crash_scenario])

# 3. Use the model to make a prediction
# Note: change 'model' to 'optimized_model' if you created it as a new variable
prediction = model.predict(input_df)[0]




print("--- AI Prediction Result ---")
print(f"Predicted Crash Severity Level: {prediction}")
probabilities = model.predict_proba(input_df)[0]
prob_dict = dict(zip(model.classes_, probabilities))
risk = prob_dict.get(1.0, 0) + 0.85*prob_dict.get(2.0,0)
if risk > 0.45:
  print("High Risk")


--- AI Prediction Result ---
Predicted Crash Severity Level: 5.0


In [26]:
probabilities = model.predict_proba(input_df)[0]

print("--- AI Confidence Breakdown ---")
for class_label, prob in zip(model.classes_, probabilities):
    print(f"Crash Level {class_label}: {prob * 100:.2f}% confidence")

--- AI Confidence Breakdown ---
Crash Level 1.0: 4.25% confidence
Crash Level 2.0: 11.04% confidence
Crash Level 3.0: 20.51% confidence
Crash Level 4.0: 28.40% confidence
Crash Level 5.0: 35.81% confidence
Crash Level 6.0: 0.00% confidence


In [8]:
df_weather = pd.read_csv('weather_analysis.csv', skiprows= 3)


df_weather['time'] = pd.to_datetime(df_weather['time'])
df_weather['year'] = df_weather['time'].dt.year
df_weather['month'] = df_weather['time'].dt.month
df_weather['day'] = df_weather['time'].dt.day
df_weather['hour'] = df_weather['time'].dt.hour

df_clean['CRASH_HOUR'] = df_clean['MILT_TIME']//100

df_unified = pd.merge(
    df_clean,
    df_weather,
    left_on=['DATE_VAL_YEAR', 'DATE_VAL_MONTH', 'DATE_VAL_DAY', 'CRASH_HOUR'],
    right_on=['year', 'month', 'day', 'hour'],
    how='left'
)
df_unified = df_unified.drop(columns=['year', 'month', 'day', 'hour', 'time'])
print(f"Stitching Complete! Your AI-ready matrix now spans {df_unified.shape} rows.")
print("New columns available for training:")
print([col for col in df_unified.columns if col not in df_clean.columns])



/tmp/ipykernel_2149/2660280869.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['CRASH_HOUR'] = df_clean['MILT_TIME']//100


Stitching Complete! Your AI-ready matrix now spans (497841, 24) rows.
New columns available for training:
['temperature_2m (°C)', 'precipitation (mm)', 'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)']
